# 02 — Theory: Collision Regularization in 1D, 2D, and 3D

This notebook is a dense theory companion for the rest of the tour. Equations are reproduced from [`theory/Regularization.md`](../../../theory/Regularization.md) with explanatory prose; for the mapping to this package's implementation see [`theory/RegularizedIntegrationDesign.md`](../../../theory/RegularizedIntegrationDesign.md).

**Goal of regularization.** Remove binary-collision singularities from pairwise $1/r$-type interactions while preserving the Hamiltonian structure, by (i) replacing the singular relative coordinate with a collision-resolving coordinate, and (ii) rescaling time so the trajectory slows down near the singularity in physical time but remains smooth in a *fictitious* time.

## 1. Singular Structure of the $N$-Body Problem

Start from a Hamiltonian

$$H(q,p,t) = \sum_{i=1}^N \frac{\|p_i\|^2}{2 m_i} + \sum_{1 \le i < j \le N} \frac{\kappa_{ij}}{r_{ij}} + R(q,p,t),\qquad r_{ij} = \|q_i - q_j\|,$$

where $\kappa_{ij}$ can have either sign and $R$ collects the remaining non-collision-singular part (for Weber this includes the velocity-dependent $\dot r^2/(2c^2)$ correction). Binary-collision singularities occur exactly where $r_{ij} = 0$.

For each close pair $(i,j)$ work in centre-of-mass and relative variables,

$$R_{ij} = \frac{m_i q_i + m_j q_j}{m_i + m_j},\qquad \vec r_{ij} = q_i - q_j,$$

$$P_{ij} = p_i + p_j,\qquad p_{ij} = \mu_{ij}\!\left(\frac{p_i}{m_i} - \frac{p_j}{m_j}\right),\qquad \mu_{ij} = \frac{m_i m_j}{m_i + m_j}.$$

The singular part lives entirely in the relative coordinate $\vec r_{ij}$, so the regularization is applied there and the centre-of-mass part is untouched.

## 2. Time Regularization in Extended Phase Space

Introduce a conjugate pair $(t, p_t)$ and a positive monitor function $g(q,p,t) > 0$. Define an *extended* Hamiltonian

$$\mathcal{K}(q, p, t, p_t) = g(q, p, t)\,\big(H(q, p, t) + p_t\big).$$

Hamilton's equations in fictitious time $\tau$ read

$$\frac{dq}{d\tau} = \frac{\partial \mathcal{K}}{\partial p},\qquad \frac{dp}{d\tau} = -\frac{\partial \mathcal{K}}{\partial q},\qquad \frac{dt}{d\tau} = g,\qquad \frac{dp_t}{d\tau} = -\frac{\partial \mathcal{K}}{\partial t}.$$

On the physical manifold $H + p_t = 0$ (enforced by choice of initial $p_t$) the equations collapse to

$$\frac{dq}{d\tau} = g\,\frac{\partial H}{\partial p},\qquad \frac{dp}{d\tau} = -g\,\frac{\partial H}{\partial q},\qquad \frac{dt}{d\tau} = g,$$

so in particular $dt = g\,d\tau$. Choosing $g \to 0$ near a collision slows physical time exactly where the vector field blows up, and the transformed vector field in $\tau$ becomes finite.

**Discretization note.** For discrete midpoint-type integrators the monitor is typically evaluated at the start of a substep and *frozen* over that substep — this keeps the physical-time increment consistent while preserving the close-encounter slowing between substeps (see [`theory/RegularizedIntegrationDesign.md`](../../../theory/RegularizedIntegrationDesign.md)).

## 3. 1D Binary Regularization

For a 1D relative coordinate $x$, define $r = |x|$ and a local chart sign $s \in \{+1, -1\}$:

$$x = s\,u^2,\qquad r = u^2.$$

Preservation of the canonical one-form $p_x\,dx = p_u\,du$ gives

$$p_u = 2 s u\, p_x,\qquad p_x = \frac{p_u}{2 s u}.$$

For a pair Hamiltonian $H_{\text{pair}} = p_x^2/(2\mu) + \kappa/|x| + R(x, p_x, t)$, the kinetic term transforms as

$$\frac{p_x^2}{2\mu} = \frac{p_u^2}{8\mu u^2}.$$

With Sundman scaling $dt = r\,d\tau = u^2\,d\tau$ and fixing the energy at $E$,

$$K_{1\mathrm{D}} = u^2\,(H_{\text{pair}} - E) = \frac{p_u^2}{8\mu} + \kappa + u^2\,(R - E),$$

which is **finite at $u = 0$**. Crossing $x = 0$ is handled by switching chart sign $s$.

## 4. 2D Levi-Civita Regularization

Let $\mathbf{r} = (x, y)$ and $u = (u_1, u_2)$. Define the map

$$x = u_1^2 - u_2^2,\qquad y = 2 u_1 u_2,$$

which in complex form is the squaring map $z = x + i y = (u_1 + i u_2)^2$. Then

$$r = \sqrt{x^2 + y^2} = u_1^2 + u_2^2 = \|u\|^2.$$

The Jacobian is

$$J = \frac{\partial(x, y)}{\partial(u_1, u_2)} = 2\begin{bmatrix} u_1 & -u_2 \\ u_2 & u_1 \end{bmatrix},\qquad J J^\top = 4 r\,I_2.$$

Canonical momenta from $p \cdot d\mathbf{r} = U \cdot du$:

$$U = J^\top p,\qquad U_1 = 2(u_1 p_x + u_2 p_y),\qquad U_2 = 2(-u_2 p_x + u_1 p_y).$$

Therefore

$$\|p\|^2 = \frac{\|U\|^2}{4 r},\qquad \frac{\|p\|^2}{2\mu} = \frac{\|U\|^2}{8 \mu r}.$$

With $dt = r\,d\tau$,

$$K_{2\mathrm{D}} = r\,(H_{\text{pair}} - E) = \frac{\|U\|^2}{8\mu} + \kappa + r\,(R - E),$$

regular at $r = 0$. The map is two-to-one: $(u_1, u_2)$ and $(-u_1, -u_2)$ represent the same physical point. This is the `:lifted_pair` backend in this package.

## 5. 3D Kustaanheimo–Stiefel Regularization

For a 3D relative coordinate $\mathbf{r} = (x_1, x_2, x_3)$, introduce $u = (u_1, u_2, u_3, u_4) \in \mathbb{R}^4$ via

$$x_1 = u_1^2 - u_2^2 - u_3^2 + u_4^2,$$
$$x_2 = 2(u_1 u_2 - u_3 u_4),$$
$$x_3 = 2(u_1 u_3 + u_2 u_4),$$

so that

$$r = \sqrt{x_1^2 + x_2^2 + x_3^2} = u_1^2 + u_2^2 + u_3^2 + u_4^2 = \|u\|^2.$$

Let $J = \partial x/\partial u \in \mathbb{R}^{3\times 4}$. It satisfies $J J^\top = 4 r\,I_3$. Define a 4D momentum

$$U = J^\top p + \lambda\,n(u),\qquad n(u) = (u_4, -u_3, u_2, -u_1)^\top,$$

and impose the **KS bilinear constraint** to remove the gauge freedom:

$$\Psi(u, U) = u_4 U_1 - u_3 U_2 + u_2 U_3 - u_1 U_4 = 0.$$

On $\Psi = 0$ we recover $U = J^\top p$ and

$$\|p\|^2 = \frac{\|U\|^2}{4 r},\qquad \frac{\|p\|^2}{2\mu} = \frac{\|U\|^2}{8\mu r}.$$

With $dt = r\,d\tau$,

$$K_{3\mathrm{D}} = r\,(H_{\text{pair}} - E) = \frac{\|U\|^2}{8\mu} + \kappa + r\,(R - E)$$

plus the additional constraint $\Psi = 0$. Binary collisions map to $u = 0$ without singular derivatives in $\tau$. **This package currently exposes KS as a diagnostic projection inside `:adaptive_cartesian` for 3D** — a true 3D lifted stepper is deferred to a future release (see [`theory/RegularizedIntegrationDesign.md`](../../../theory/RegularizedIntegrationDesign.md)).

## 6. Lifting to $N$-Body Systems

For $N > 2$, regularization is applied to *close binary subsystems* in relative coordinates, then coupled back to the rest of the system. Global time scaling uses a positive $g$ depending on all pair distances. Common families:

$$dt = g\,d\tau,\qquad g = r_{ab}\ \text{(single active pair)},$$

$$g = \left(\sum_{i < j} \frac{w_{ij}}{r_{ij}}\right)^{-1}\ \text{(global close-encounter monitor)},$$

$$g = \frac{1}{\alpha T + \beta \Omega + \gamma},\qquad \Omega = \sum_{i < j} \frac{w_{ij}}{r_{ij}},\qquad \alpha, \beta, \gamma \ge 0.$$

When multiple KS pairs are active, the constraints $\Psi_a(Q, P) = 0$ are enforced for each pair.

**Chain coordinates.** When several nearby bodies interact simultaneously, direct pairwise relative coordinates become ill-conditioned. The standard remedy is to choose an ordered chain of bodies $i_1, \dots, i_M$ through the closest separations, use link vectors $s_k = q_{i_{k+1}} - q_{i_k}$ as primary relatives, regularize each short link, and evolve with a global $g$. This package exposes `chain_enabled` on `RegularizationOptions` to toggle this path for encounter-graph components with more than two particles.

## 7. Weber Like-Charge Collisions — a Special Case

Weber's velocity-dependent $1 - \dot r^2/(2c^2)$ correction creates a qualitatively new phenomenon for **like charges** inside a critical radius $\rho$ where the effective force flips sign (the velocity term beats the Coulomb repulsion). Whether the resulting collision is regularizable depends on angular momentum:

- **Head-on collisions ($\ell = 0$).** The relative coordinate reaches the origin at finite speed $\sqrt{2}\,c$ and is $C^0$-continuable. The standard regularization techniques above apply, and this package additionally offers a **collision bounce** (pre-step reflection $q_{\text{rel}} \mapsto -q_{\text{rel}}$ with momenta unchanged) as an exact energy-preserving $C^0$ continuation. See [`04_close_encounters_and_bounce.ipynb`](04_close_encounters_and_bounce.ipynb).
- **Spiraling collisions ($\ell \ne 0$).** The relative coordinate reaches the origin at *infinite* speed in finite physical time, with infinite winding number. This is **not regularizable by any smooth coordinate-time transform** — the obstruction is topological. Frauenfelder & Weber (2024, Theorem 2.1) prove this, and seven regularization approaches were tested numerically (Sundman at various orders, Levi-Civita, McGehee blow-up, partial radial regularization, Birkhoff inversion, logarithmic map) — all fail for $\ell \ne 0$. See [`research/investigations/AngularMomentumRegularization.md`](../../investigations/AngularMomentumRegularization.md).

## 8. Package Implementation Map

From [`theory/RegularizedIntegrationDesign.md`](../../../theory/RegularizedIntegrationDesign.md):

| Concept (above) | Package backend | Dim support | Notes |
|---|---|---|---|
| §4 Levi-Civita 2D | `:lifted_pair` | 2D only | Explicit A–B–A split (perturbation + lifted pair substeps); monitor $g = \max(r, g_{\text{floor}})$ frozen per substep |
| §3/§5 as adaptive substeps | `:adaptive_cartesian` | 1D/2D/3D | Adaptive Cartesian substep count $\lceil r_{\text{on}}/\max(r, g_{\text{floor}})\rceil$, projected kernel; 3D also applies the KS bilinear projection as a diagnostic |
| §6 chain coordinates | chain mode | 1D/2D/3D | Component size $>2$ and `chain_enabled=true`; monitor $g = \max(1/\Omega, g_{\text{floor}})$ |
| §7 $\ell = 0$ bounce | `collision_bounce_radius` | all | Pre-step reflection; may be combined with `enabled=false` |

**Encounter dispatch.** Each outer step computes all pair distances, builds the encounter graph for pairs with $r \le r_{\text{on}}$, selects the connected component containing the minimum-distance pair, and uses **hysteresis** between $r_{\text{on}}$ and $r_{\text{off}}$ to avoid mode-hopping. The anchor pair that triggered activation is held fixed for the duration of the active period.

**Diagnostics you can read after `solve`.** `sol.regularization` returns a `RegularizationDiagnostics` with `requested_backend`, `used_backend`, `pair_steps`, `adaptive_pair_steps`, `lifted_pair_steps`, `chain_steps`, `unregularized_steps`, `backend_fallback_steps`, `total_substeps`, `max_substeps_used`, `activation_count`, `deactivation_count`, `min_encounter_distance`, `max_constraint_violation`, and a per-step `mode_history` vector (`0` = none, `1` = pair, `2` = chain). Every subsequent notebook in this tour reads these.

In [1]:
include(joinpath(@__DIR__, "common.jl"))

In [2]:
using Pkg
println("WeberElectrodynamics version: ", Pkg.TOML.parsefile(joinpath(dirname(dirname(pathof(WeberElectrodynamics))), "Project.toml"))["version"])

WeberElectrodynamics version: 0.4.1
